# 04 — Modelo Linear Regression

Este notebook treina modelos de **Linear Regression** usando **PySpark MLlib** para prever temperatura no estado de São Paulo.

Serão treinados dois modelos:

1. **Previsão da temperatura média de amanhã**.
2. **Previsão da temperatura média dos próximos 7 dias**.

As bases utilizadas aqui já foram geradas no notebook de pré-processamento e estão salvas em formato **Parquet**. Elas já passaram por:

- limpeza e padronização dos dados;
- tratamento de valores sentinela;
- aplicação de regras físicas;
- criação de variáveis temporais e geográficas;
- agregação diária por estação;
- criação dos alvos de previsão;
- separação temporal entre treino e teste;
- imputação sem vazamento de dados;
- indexação das variáveis categóricas.

A separação treino/teste segue uma lógica temporal:

- **Treino:** anos anteriores a 2018;
- **Teste:** anos de 2018 em diante.

Assim como no notebook de Random Forest, este notebook mantém o mesmo fluxo de avaliação e visualização:

- uso de **MLlib** para o modelo;
- uso de **Spark SQL** para consultas, agregações, joins e análises;
- uso de **Plotly** para gráficos;
- sem uso de `toPandas()`.

A diferença principal é o algoritmo: aqui será usado o modelo **LinearRegression**.

## 1. Imports e inicialização da SparkSession

Nesta etapa são importadas as bibliotecas necessárias para:

- carregar os Parquets;
- montar o vetor de features;
- treinar o modelo Linear Regression com MLlib;
- avaliar regressão;
- gerar gráficos interativos com Plotly.

As transformações analíticas serão feitas preferencialmente com `spark.sql`.

In [1]:
from pyspark.sql import SparkSession

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

import plotly.express as px
import plotly.graph_objects as go

In [2]:
spark = (
    SparkSession.builder
    .appName("04_modelo_linear_regression")
    .getOrCreate()
)

spark

## 2. Caminhos das bases Parquet

As bases finais já foram criadas no notebook de pré-processamento.

Aqui serão carregados os datasets separados de treino e teste para os dois objetivos:

- previsão da temperatura média de amanhã;
- previsão da temperatura média dos próximos 7 dias.

Também serão carregadas as bases completas, porque elas ainda possuem colunas interpretáveis como `tipo_area`, `macro_regiao_sp` e `faixa_altitude`. Essas colunas serão usadas depois para análises e gráficos por tipo de região.

In [3]:
amanha_train_path = "/home/jovyan/work/data/processed/weather_sp_amanha_train"
amanha_test_path  = "/home/jovyan/work/data/processed/weather_sp_amanha_test"

semana_train_path = "/home/jovyan/work/data/processed/weather_sp_semana_train"
semana_test_path  = "/home/jovyan/work/data/processed/weather_sp_semana_test"

dataset_amanha_path = "/home/jovyan/work/data/processed/weather_sp_dataset_amanha"
dataset_semana_path = "/home/jovyan/work/data/processed/weather_sp_dataset_semana"

modelos_path = "/home/jovyan/work/models"
resultados_path = "/home/jovyan/work/data/processed"

## 3. Carregamento das bases

As bases são lidas diretamente do formato Parquet.

Depois da leitura, cada DataFrame é registrado como uma view temporária para permitir consultas com `spark.sql`.

In [4]:
amanha_train = spark.read.parquet(amanha_train_path)
amanha_test = spark.read.parquet(amanha_test_path)

semana_train = spark.read.parquet(semana_train_path)
semana_test = spark.read.parquet(semana_test_path)

dataset_amanha_completo = spark.read.parquet(dataset_amanha_path)
dataset_semana_completo = spark.read.parquet(dataset_semana_path)

amanha_train.createOrReplaceTempView("amanha_train")
amanha_test.createOrReplaceTempView("amanha_test")

semana_train.createOrReplaceTempView("semana_train")
semana_test.createOrReplaceTempView("semana_test")

dataset_amanha_completo.createOrReplaceTempView("dataset_amanha_completo")
dataset_semana_completo.createOrReplaceTempView("dataset_semana_completo")

In [5]:
spark.sql("""
    SELECT 'amanha_train' AS base, COUNT(*) AS total_linhas FROM amanha_train
    UNION ALL
    SELECT 'amanha_test' AS base, COUNT(*) AS total_linhas FROM amanha_test
    UNION ALL
    SELECT 'semana_train' AS base, COUNT(*) AS total_linhas FROM semana_train
    UNION ALL
    SELECT 'semana_test' AS base, COUNT(*) AS total_linhas FROM semana_test
""").show(truncate=False)

+------------+------------+
|base        |total_linhas|
+------------+------------+
|amanha_train|120232      |
|amanha_test |45940       |
|semana_train|120232      |
|semana_test |45940       |
+------------+------------+



## 4. Cache estratégico dos datasets de modelagem

No pré-processamento, o cache foi evitado para reduzir consumo de memória.

Aqui, no notebook de modelagem, o cache faz sentido porque os mesmos datasets serão usados várias vezes:

- treinamento;
- avaliação;
- geração de previsões;
- análise de erro;
- gráficos.

O `count()` logo após o `cache()` força a materialização dos dados em memória.

In [6]:
amanha_train = amanha_train.cache()
amanha_test = amanha_test.cache()

semana_train = semana_train.cache()
semana_test = semana_test.cache()

amanha_train.count()
amanha_test.count()
semana_train.count()
semana_test.count()

amanha_train.createOrReplaceTempView("amanha_train")
amanha_test.createOrReplaceTempView("amanha_test")
semana_train.createOrReplaceTempView("semana_train")
semana_test.createOrReplaceTempView("semana_test")

## 5. Conferência dos schemas

Antes de montar o modelo, é importante verificar se os datasets possuem:

- colunas de identificação;
- features numéricas já imputadas;
- variáveis categóricas indexadas;
- coluna-alvo correta.

In [7]:
amanha_train.printSchema()

root
 |-- station: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- data_formatada: date (nullable = true)
 |-- ano_imputado: integer (nullable = true)
 |-- mes_sin_imputado: double (nullable = true)
 |-- mes_cos_imputado: double (nullable = true)
 |-- latitude_imputado: double (nullable = true)
 |-- longitude_imputado: double (nullable = true)
 |-- altitude_imputado: double (nullable = true)
 |-- temp_media_dia_imputado: double (nullable = true)
 |-- temp_min_dia_imputado: double (nullable = true)
 |-- temp_max_dia_imputado: double (nullable = true)
 |-- temp_orvalho_media_dia_imputado: double (nullable = true)
 |-- umidade_media_dia_imputado: double (nullable = true)
 |-- umidade_min_dia_imputado: double (nullable = true)
 |-- umidade_max_dia_imputado: double (nullable = true)
 |-- pressao_media_dia_imputado: double (nullable = true)
 |-- precipitacao_total_dia_imputado: double (nullable = true)
 |-- radiacao_media_dia_imputado: double (nullable = true)
 |-- 

In [8]:
semana_train.printSchema()

root
 |-- station: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- data_formatada: date (nullable = true)
 |-- ano_imputado: integer (nullable = true)
 |-- mes_sin_imputado: double (nullable = true)
 |-- mes_cos_imputado: double (nullable = true)
 |-- latitude_imputado: double (nullable = true)
 |-- longitude_imputado: double (nullable = true)
 |-- altitude_imputado: double (nullable = true)
 |-- temp_media_dia_imputado: double (nullable = true)
 |-- temp_min_dia_imputado: double (nullable = true)
 |-- temp_max_dia_imputado: double (nullable = true)
 |-- temp_orvalho_media_dia_imputado: double (nullable = true)
 |-- umidade_media_dia_imputado: double (nullable = true)
 |-- umidade_min_dia_imputado: double (nullable = true)
 |-- umidade_max_dia_imputado: double (nullable = true)
 |-- pressao_media_dia_imputado: double (nullable = true)
 |-- precipitacao_total_dia_imputado: double (nullable = true)
 |-- radiacao_media_dia_imputado: double (nullable = true)
 |-- 

## 6. Definição dos alvos e das features

O modelo Linear Regression do MLlib recebe as variáveis explicativas em uma única coluna vetorial chamada `features`.

Como o pré-processamento já deixou as colunas prontas para modelagem, as features serão identificadas automaticamente, removendo apenas:

- colunas de identificação;
- coluna-alvo.

As colunas de identificação não entram no modelo porque representam nomes, códigos ou datas, e não variáveis numéricas explicativas diretas.

In [9]:
coluna_alvo_amanha = "temperatura_amanha"
coluna_alvo_semana = "temperatura_media_proximos_7_dias"

colunas_identificacao = [
    "station",
    "station_code",
    "data_formatada"
]

features_amanha = [
    c for c in amanha_train.columns
    if c not in colunas_identificacao + [coluna_alvo_amanha]
]

features_semana = [
    c for c in semana_train.columns
    if c not in colunas_identificacao + [coluna_alvo_semana]
]

print(f"Features amanhã ({len(features_amanha)}):")
print(features_amanha)

print(f"\nFeatures semana ({len(features_semana)}):")
print(features_semana)

Features amanhã (26):
['ano_imputado', 'mes_sin_imputado', 'mes_cos_imputado', 'latitude_imputado', 'longitude_imputado', 'altitude_imputado', 'temp_media_dia_imputado', 'temp_min_dia_imputado', 'temp_max_dia_imputado', 'temp_orvalho_media_dia_imputado', 'umidade_media_dia_imputado', 'umidade_min_dia_imputado', 'umidade_max_dia_imputado', 'pressao_media_dia_imputado', 'precipitacao_total_dia_imputado', 'radiacao_media_dia_imputado', 'vento_medio_dia_imputado', 'rajada_max_dia_imputado', 'temp_media_ontem_imputado', 'temp_media_ultimos_3_dias_imputado', 'temp_media_ultimos_7_dias_imputado', 'umidade_media_ultimos_7_dias_imputado', 'precipitacao_ultimos_7_dias_imputado', 'macro_regiao_sp_idx', 'tipo_area_idx', 'faixa_altitude_idx']

Features semana (26):
['ano_imputado', 'mes_sin_imputado', 'mes_cos_imputado', 'latitude_imputado', 'longitude_imputado', 'altitude_imputado', 'temp_media_dia_imputado', 'temp_min_dia_imputado', 'temp_max_dia_imputado', 'temp_orvalho_media_dia_imputado', 'umi

## 7. Por que Linear Regression?

A **Regressão Linear** é um modelo supervisionado usado para prever uma variável numérica contínua.

Ela é interessante para este projeto porque:

- é simples de interpretar;
- serve como uma boa base de comparação com modelos mais complexos;
- permite observar relações lineares entre variáveis climáticas e temperatura futura;
- costuma ter bom desempenho quando a variável-alvo depende fortemente de tendências recentes, como médias de temperatura dos últimos dias.

Um ponto importante: a Regressão Linear também **não entende sequência temporal sozinha**. Por isso, o pré-processamento criou features de defasagem e janelas móveis, como:

- temperatura média de ontem;
- média dos últimos 3 dias;
- média dos últimos 7 dias;
- umidade média dos últimos 7 dias;
- precipitação acumulada dos últimos 7 dias.

Assim, o modelo recebe contexto temporal em forma de colunas explicativas.

Como as bases já vêm prontas do pré-processamento, este notebook mantém o mesmo esquema do Random Forest e troca apenas o algoritmo para `LinearRegression`.

## 8. Treinamento do modelo para temperatura de amanhã

Nesta etapa será treinado o primeiro modelo:

> prever a `temperatura_amanha`.

O `VectorAssembler` junta todas as features em uma coluna vetorial, e o `LinearRegression` realiza o treinamento.

In [10]:
assembler_amanha = VectorAssembler(
    inputCols=features_amanha,
    outputCol="features",
    handleInvalid="keep"
)

lr_amanha = LinearRegression(
    featuresCol="features",
    labelCol=coluna_alvo_amanha,
    predictionCol="prediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline_amanha = Pipeline(stages=[
    assembler_amanha,
    lr_amanha
])

In [11]:
modelo_lr_amanha = pipeline_amanha.fit(amanha_train)

pred_amanha = modelo_lr_amanha.transform(amanha_test)
pred_amanha.createOrReplaceTempView("pred_amanha")

print("Modelo Linear Regression para temperatura de amanhã treinado com sucesso.")

Modelo Linear Regression para temperatura de amanhã treinado com sucesso.


## 9. Previsões do modelo de amanhã

Após o treinamento, o modelo é aplicado à base de teste.

Também é criada a coluna `erro_absoluto`, que representa a diferença absoluta entre o valor real e o valor previsto.

Essa criação é feita com `spark.sql`.

In [12]:
pred_amanha = spark.sql(f"""
    SELECT
        *,
        ABS({coluna_alvo_amanha} - prediction) AS erro_absoluto
    FROM pred_amanha
""")

pred_amanha.createOrReplaceTempView("pred_amanha")

spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND({coluna_alvo_amanha}, 2) AS real,
        ROUND(prediction, 2) AS previsto,
        ROUND(erro_absoluto, 2) AS erro_abs
    FROM pred_amanha
    LIMIT 20
""").show(truncate=False)

+--------+------------+--------------+-----+--------+--------+
|station |station_code|data_formatada|real |previsto|erro_abs|
+--------+------------+--------------+-----+--------+--------+
|SOROCABA|A713        |2018-01-01    |22.82|22.53   |0.29    |
|SOROCABA|A713        |2018-01-02    |22.72|22.59   |0.13    |
|SOROCABA|A713        |2018-01-03    |23.15|22.7    |0.45    |
|SOROCABA|A713        |2018-01-04    |23.26|23.17   |0.09    |
|SOROCABA|A713        |2018-01-05    |22.95|23.83   |0.88    |
|SOROCABA|A713        |2018-01-06    |21.51|23.21   |1.7     |
|SOROCABA|A713        |2018-01-07    |20.7 |21.37   |0.67    |
|SOROCABA|A713        |2018-01-08    |20.71|21.37   |0.66    |
|SOROCABA|A713        |2018-01-09    |23.67|21.74   |1.93    |
|SOROCABA|A713        |2018-01-10    |22.5 |24.26   |1.76    |
|SOROCABA|A713        |2018-01-11    |20.55|22.42   |1.87    |
|SOROCABA|A713        |2018-01-12    |21.09|21.14   |0.05    |
|SOROCABA|A713        |2018-01-13    |22.45|21.6    |0.

## 10. Função de avaliação de regressão

Serão usadas três métricas:

- **MAE:** erro médio absoluto em °C. É a métrica mais fácil de interpretar.
- **RMSE:** penaliza erros grandes com mais intensidade.
- **R²:** indica a proporção da variação explicada pelo modelo.

Quanto menores MAE e RMSE, melhor. Quanto mais próximo de 1 o R², melhor.

A avaliação usa `RegressionEvaluator`, que é um componente próprio do MLlib.

In [13]:
def avaliar_regressao(predicoes, coluna_alvo, nome_modelo):
    avaliador_mae = RegressionEvaluator(
        labelCol=coluna_alvo,
        predictionCol="prediction",
        metricName="mae"
    )

    avaliador_rmse = RegressionEvaluator(
        labelCol=coluna_alvo,
        predictionCol="prediction",
        metricName="rmse"
    )

    avaliador_r2 = RegressionEvaluator(
        labelCol=coluna_alvo,
        predictionCol="prediction",
        metricName="r2"
    )

    mae = avaliador_mae.evaluate(predicoes)
    rmse = avaliador_rmse.evaluate(predicoes)
    r2 = avaliador_r2.evaluate(predicoes)

    print(nome_modelo)
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

    return {
        "modelo": nome_modelo,
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2)
    }

In [14]:
metricas_amanha = avaliar_regressao(
    pred_amanha,
    coluna_alvo_amanha,
    "Linear Regression - Temperatura amanhã"
)

Linear Regression - Temperatura amanhã
MAE : 1.1613
RMSE: 1.5695
R²  : 0.8307


## 11. Treinamento do modelo para média dos próximos 7 dias

Agora será treinado o segundo modelo:

> prever a `temperatura_media_proximos_7_dias`.

Este alvo representa a temperatura média dos sete dias seguintes para cada estação meteorológica.

In [15]:
assembler_semana = VectorAssembler(
    inputCols=features_semana,
    outputCol="features",
    handleInvalid="keep"
)

lr_semana = LinearRegression(
    featuresCol="features",
    labelCol=coluna_alvo_semana,
    predictionCol="prediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline_semana = Pipeline(stages=[
    assembler_semana,
    lr_semana
])

In [16]:
modelo_lr_semana = pipeline_semana.fit(semana_train)

pred_semana = modelo_lr_semana.transform(semana_test)
pred_semana.createOrReplaceTempView("pred_semana")

print("Modelo Linear Regression para temperatura média dos próximos 7 dias treinado com sucesso.")

Modelo Linear Regression para temperatura média dos próximos 7 dias treinado com sucesso.


## 12. Previsões do modelo de próximos 7 dias

A coluna de erro absoluto também é criada com `spark.sql`.

In [17]:
pred_semana = spark.sql(f"""
    SELECT
        *,
        ABS({coluna_alvo_semana} - prediction) AS erro_absoluto
    FROM pred_semana
""")

pred_semana.createOrReplaceTempView("pred_semana")

spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND({coluna_alvo_semana}, 2) AS real,
        ROUND(prediction, 2) AS previsto,
        ROUND(erro_absoluto, 2) AS erro_abs
    FROM pred_semana
    LIMIT 20
""").show(truncate=False)

+--------+------------+--------------+-----+--------+--------+
|station |station_code|data_formatada|real |previsto|erro_abs|
+--------+------------+--------------+-----+--------+--------+
|SOROCABA|A713        |2018-01-01    |22.44|22.88   |0.44    |
|SOROCABA|A713        |2018-01-02    |22.14|22.88   |0.74    |
|SOROCABA|A713        |2018-01-03    |22.28|23.11   |0.83    |
|SOROCABA|A713        |2018-01-04    |22.19|23.82   |1.63    |
|SOROCABA|A713        |2018-01-05    |21.8 |23.86   |2.07    |
|SOROCABA|A713        |2018-01-06    |21.53|23.36   |1.83    |
|SOROCABA|A713        |2018-01-07    |21.67|22.74   |1.07    |
|SOROCABA|A713        |2018-01-08    |22.07|22.68   |0.61    |
|SOROCABA|A713        |2018-01-09    |22.37|22.73   |0.36    |
|SOROCABA|A713        |2018-01-10    |22.52|23.74   |1.23    |
|SOROCABA|A713        |2018-01-11    |22.88|22.9    |0.02    |
|SOROCABA|A713        |2018-01-12    |23.35|22.23   |1.12    |
|SOROCABA|A713        |2018-01-13    |23.71|22.28   |1.

In [18]:
metricas_semana = avaliar_regressao(
    pred_semana,
    coluna_alvo_semana,
    "Linear Regression - Temperatura média próximos 7 dias"
)

Linear Regression - Temperatura média próximos 7 dias
MAE : 1.2861
RMSE: 1.6738
R²  : 0.7614


## 13. Comparação final das métricas

Nesta etapa, as métricas dos dois modelos são reunidas em uma única tabela.

Essa comparação ajuda a entender se o modelo tem melhor desempenho prevendo o dia seguinte ou a média dos próximos sete dias.

In [19]:
metricas_lr = spark.createDataFrame([
    metricas_amanha,
    metricas_semana
])

metricas_lr.createOrReplaceTempView("metricas_lr")

spark.sql("""
    SELECT
        modelo,
        ROUND(mae, 4) AS mae,
        ROUND(rmse, 4) AS rmse,
        ROUND(r2, 4) AS r2
    FROM metricas_lr
""").show(truncate=False)

+-----------------------------------------------------+------+------+------+
|modelo                                               |mae   |rmse  |r2    |
+-----------------------------------------------------+------+------+------+
|Linear Regression - Temperatura amanhã               |1.1613|1.5695|0.8307|
|Linear Regression - Temperatura média próximos 7 dias|1.2861|1.6738|0.7614|
+-----------------------------------------------------+------+------+------+



### Função auxiliar para gráficos Plotly sem `toPandas`

Para manter o notebook coerente com Spark, os gráficos serão criados a partir de listas obtidas com `collect()` em resultados pequenos e já agregados.

Não será usado `toPandas()`.

In [20]:
def spark_df_para_dicts(df):
    return [row.asDict() for row in df.collect()]

def coluna_lista(linhas, coluna):
    return [linha[coluna] for linha in linhas]

### Gráfico: comparação do MAE

O MAE é especialmente útil para comunicação do resultado, pois pode ser lido diretamente como erro médio em graus Celsius.

In [21]:
metricas_plot_df = spark.sql("""
    SELECT
        modelo,
        mae
    FROM metricas_lr
    ORDER BY modelo
""")

metricas_plot = spark_df_para_dicts(metricas_plot_df)

fig = px.bar(
    metricas_plot,
    x="modelo",
    y="mae",
    text="mae",
    title="Comparação do MAE entre os modelos Linear Regression",
    labels={
        "modelo": "Modelo",
        "mae": "MAE (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Se o modelo de próximos 7 dias apresentar MAE menor do que o modelo de amanhã, isso pode indicar que médias semanais suavizam oscilações diárias.

Em outras palavras, prever uma média de 7 dias pode ser mais estável do que prever exatamente o comportamento do dia seguinte, que pode sofrer influência de mudanças bruscas de tempo.

## 14. Coeficientes do modelo

Diferente do Random Forest, que fornece importância das variáveis, a Regressão Linear fornece **coeficientes**.

Os coeficientes indicam a direção e a intensidade da relação linear entre cada feature e o alvo, considerando o conjunto de variáveis usado no modelo.

Como as features possuem escalas diferentes, os coeficientes devem ser interpretados com cuidado. Ainda assim, eles ajudam a identificar quais variáveis têm maior peso linear no modelo.

In [22]:
def obter_coeficientes(modelo_pipeline, lista_features):
    modelo_lr = modelo_pipeline.stages[-1]
    coeficientes = modelo_lr.coefficients
    intercepto = float(modelo_lr.intercept)

    linhas = [
        (feature, float(coeficiente), abs(float(coeficiente)))
        for feature, coeficiente in zip(lista_features, coeficientes)
    ]

    df_coeficientes = spark.createDataFrame(
        linhas,
        ["feature", "coeficiente", "coeficiente_absoluto"]
    )

    return df_coeficientes, intercepto

In [23]:
coef_amanha, intercepto_amanha = obter_coeficientes(modelo_lr_amanha, features_amanha)
coef_semana, intercepto_semana = obter_coeficientes(modelo_lr_semana, features_semana)

coef_amanha.createOrReplaceTempView("coef_amanha")
coef_semana.createOrReplaceTempView("coef_semana")

print(f"Intercepto modelo amanhã: {intercepto_amanha:.4f}")
print(f"Intercepto modelo próximos 7 dias: {intercepto_semana:.4f}")

print("Coeficientes - modelo amanhã")
spark.sql("""
    SELECT
        feature,
        ROUND(coeficiente, 6) AS coeficiente,
        ROUND(coeficiente_absoluto, 6) AS coeficiente_absoluto
    FROM coef_amanha
    ORDER BY coeficiente_absoluto DESC
    LIMIT 30
""").show(30, truncate=False)

print("Coeficientes - modelo próximos 7 dias")
spark.sql("""
    SELECT
        feature,
        ROUND(coeficiente, 6) AS coeficiente,
        ROUND(coeficiente_absoluto, 6) AS coeficiente_absoluto
    FROM coef_semana
    ORDER BY coeficiente_absoluto DESC
    LIMIT 30
""").show(30, truncate=False)

Intercepto modelo amanhã: -11.7825
Intercepto modelo próximos 7 dias: -28.7250
Coeficientes - modelo amanhã
+-------------------------------------+-----------+--------------------+
|feature                              |coeficiente|coeficiente_absoluto|
+-------------------------------------+-----------+--------------------+
|mes_cos_imputado                     |0.7606     |0.7606              |
|temp_media_dia_imputado              |0.752417   |0.752417            |
|tipo_area_idx                        |0.437833   |0.437833            |
|mes_sin_imputado                     |0.281895   |0.281895            |
|temp_max_dia_imputado                |0.21276    |0.21276             |
|temp_media_ontem_imputado            |-0.178359  |0.178359            |
|faixa_altitude_idx                   |-0.16064   |0.16064             |
|latitude_imputado                    |0.155537   |0.155537            |
|temp_media_ultimos_7_dias_imputado   |0.142556   |0.142556            |
|rajada_max_dia_

### Gráfico: top 15 coeficientes absolutos — amanhã

In [24]:
coef_amanha_top15 = spark.sql("""
    SELECT
        feature,
        coeficiente,
        coeficiente_absoluto
    FROM coef_amanha
    ORDER BY coeficiente_absoluto DESC
    LIMIT 15
""")

coef_amanha_plot = spark_df_para_dicts(coef_amanha_top15)

fig = px.bar(
    coef_amanha_plot,
    x="coeficiente_absoluto",
    y="feature",
    orientation="h",
    title="Top 15 coeficientes absolutos — previsão de amanhã",
    labels={
        "coeficiente_absoluto": "Coeficiente absoluto",
        "feature": "Variável"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### Gráfico: top 15 coeficientes absolutos — próximos 7 dias

In [25]:
coef_semana_top15 = spark.sql("""
    SELECT
        feature,
        coeficiente,
        coeficiente_absoluto
    FROM coef_semana
    ORDER BY coeficiente_absoluto DESC
    LIMIT 15
""")

coef_semana_plot = spark_df_para_dicts(coef_semana_top15)

fig = px.bar(
    coef_semana_plot,
    x="coeficiente_absoluto",
    y="feature",
    orientation="h",
    title="Top 15 coeficientes absolutos — previsão dos próximos 7 dias",
    labels={
        "coeficiente_absoluto": "Coeficiente absoluto",
        "feature": "Variável"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### Insight esperado

As variáveis de temperatura recente tendem a ter forte peso no modelo, como:

- `temp_media_dia_imputado`;
- `temp_media_ontem_imputado`;
- `temp_media_ultimos_3_dias_imputado`;
- `temp_media_ultimos_7_dias_imputado`.

Isso é coerente com o comportamento climático: a temperatura futura depende fortemente das condições térmicas recentes.

Como a Regressão Linear é sensível à escala das variáveis e trabalha com relações lineares, os coeficientes não devem ser lidos exatamente como “importância” no mesmo sentido do Random Forest. Eles indicam peso linear dentro da equação do modelo.

## 15. Recuperação das informações geográficas interpretáveis

As bases finais de treino e teste foram salvas com as features já prontas para modelagem.

Para criar gráficos e insights mais interpretáveis, vamos recuperar da base completa algumas colunas geográficas:

- `tipo_area`;
- `macro_regiao_sp`;
- `faixa_altitude`;
- `altitude`;
- `latitude`;
- `longitude`.

Essa é a opção escolhida para permitir análises por urbano/metropolitano, litoral e serra/altitude sem mexer novamente no pré-processamento.

In [26]:
contexto_amanha = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        altitude,
        latitude,
        longitude
    FROM dataset_amanha_completo
""")

contexto_semana = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        altitude,
        latitude,
        longitude
    FROM dataset_semana_completo
""")

contexto_amanha.createOrReplaceTempView("contexto_amanha")
contexto_semana.createOrReplaceTempView("contexto_semana")

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `tipo_area` cannot be resolved. Did you mean one of the following? [`tipo_area_idx`, `station`, `station_code`, `ano_imputado`, `data_formatada`].; line 5 pos 8;
'Distinct
+- 'Project [station_code#241, data_formatada#242, 'tipo_area, 'macro_regiao_sp, 'faixa_altitude, 'altitude, 'latitude, 'longitude]
   +- SubqueryAlias dataset_amanha_completo
      +- View (`dataset_amanha_completo`, [station#240,station_code#241,data_formatada#242,ano_imputado#243,mes_sin_imputado#244,mes_cos_imputado#245,latitude_imputado#246,longitude_imputado#247,altitude_imputado#248,temp_media_dia_imputado#249,temp_min_dia_imputado#250,temp_max_dia_imputado#251,temp_orvalho_media_dia_imputado#252,umidade_media_dia_imputado#253,umidade_min_dia_imputado#254,umidade_max_dia_imputado#255,pressao_media_dia_imputado#256,precipitacao_total_dia_imputado#257,radiacao_media_dia_imputado#258,vento_medio_dia_imputado#259,rajada

In [ ]:
pred_amanha_ctx = spark.sql("""
    SELECT
        p.*,
        c.tipo_area,
        c.macro_regiao_sp,
        c.faixa_altitude,
        c.altitude,
        c.latitude,
        c.longitude
    FROM pred_amanha p
    LEFT JOIN contexto_amanha c
        ON p.station_code = c.station_code
       AND p.data_formatada = c.data_formatada
""")

pred_semana_ctx = spark.sql("""
    SELECT
        p.*,
        c.tipo_area,
        c.macro_regiao_sp,
        c.faixa_altitude,
        c.altitude,
        c.latitude,
        c.longitude
    FROM pred_semana p
    LEFT JOIN contexto_semana c
        ON p.station_code = c.station_code
       AND p.data_formatada = c.data_formatada
""")

pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")
pred_semana_ctx.createOrReplaceTempView("pred_semana_ctx")

In [ ]:
spark.sql(f"""
    SELECT
        station,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        ROUND({coluna_alvo_amanha}, 2) AS real,
        ROUND(prediction, 2) AS previsto,
        ROUND(erro_absoluto, 2) AS erro_abs
    FROM pred_amanha_ctx
    LIMIT 20
""").show(truncate=False)

## 16. Erro médio por tipo de área

Nesta seção, o erro do modelo é analisado por tipo de área.

Essa análise é importante porque diferentes regiões podem ter comportamentos climáticos distintos:

- áreas urbanas podem sofrer influência de ilha de calor;
- áreas litorâneas tendem a ter maior influência da umidade e do oceano;
- regiões de serra/altitude podem apresentar temperaturas mais baixas e variações específicas;
- áreas do interior podem ter maior amplitude térmica em alguns períodos.

In [ ]:
erro_tipo_area_amanha = spark.sql(f"""
    SELECT
        tipo_area,
        COUNT(*) AS total_registros,
        ROUND(AVG({coluna_alvo_amanha}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_amanha_ctx
    GROUP BY tipo_area
    ORDER BY tipo_area
""")

erro_tipo_area_semana = spark.sql(f"""
    SELECT
        tipo_area,
        COUNT(*) AS total_registros,
        ROUND(AVG({coluna_alvo_semana}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_semana_ctx
    GROUP BY tipo_area
    ORDER BY tipo_area
""")

erro_tipo_area_amanha.createOrReplaceTempView("erro_tipo_area_amanha")
erro_tipo_area_semana.createOrReplaceTempView("erro_tipo_area_semana")

erro_tipo_area_amanha.show(truncate=False)
erro_tipo_area_semana.show(truncate=False)

### Gráfico: MAE por tipo de área — amanhã

In [ ]:
erro_tipo_area_amanha_plot = spark_df_para_dicts(erro_tipo_area_amanha)

fig = px.bar(
    erro_tipo_area_amanha_plot,
    x="tipo_area",
    y="mae",
    text="mae",
    title="Erro médio absoluto por tipo de área — previsão de amanhã",
    labels={
        "tipo_area": "Tipo de área",
        "mae": "MAE (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Gráfico: MAE por tipo de área — próximos 7 dias

In [ ]:
erro_tipo_area_semana_plot = spark_df_para_dicts(erro_tipo_area_semana)

fig = px.bar(
    erro_tipo_area_semana_plot,
    x="tipo_area",
    y="mae",
    text="mae",
    title="Erro médio absoluto por tipo de área — previsão dos próximos 7 dias",
    labels={
        "tipo_area": "Tipo de área",
        "mae": "MAE (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Se o erro for maior em `serra_altitude`, isso pode indicar que a dinâmica térmica dessas áreas é mais específica por causa do relevo e da altitude.

Se o erro for maior em `litoral`, pode haver influência de umidade, brisa marítima e menor amplitude térmica, que talvez não estejam totalmente representadas pelas features disponíveis.

Se o erro for maior em `urbano_metropolitano`, pode ser um indício de que efeitos locais, como concentração urbana e ilha de calor, são relevantes para a previsão.

## 17. Real vs previsto por tipo de área

Este gráfico compara a temperatura real com a temperatura prevista.

Quanto mais próximos os pontos estiverem da linha ideal, melhor é o desempenho do modelo.

- Pontos acima da linha: o modelo superestimou a temperatura.
- Pontos abaixo da linha: o modelo subestimou a temperatura.

Para manter o gráfico leve, a consulta abaixo usa uma amostra limitada da base de teste.

In [ ]:
comparacao_tipo_area_amanha = spark.sql(f"""
    SELECT
        tipo_area,
        {coluna_alvo_amanha} AS real,
        prediction AS previsto
    FROM pred_amanha_ctx
    WHERE tipo_area IS NOT NULL
      AND {coluna_alvo_amanha} IS NOT NULL
      AND prediction IS NOT NULL
    LIMIT 5000
""")

comparacao_tipo_area_amanha_plot = spark_df_para_dicts(comparacao_tipo_area_amanha)

valores_reais = coluna_lista(comparacao_tipo_area_amanha_plot, "real")
valores_previstos = coluna_lista(comparacao_tipo_area_amanha_plot, "previsto")

min_val = min(min(valores_reais), min(valores_previstos))
max_val = max(max(valores_reais), max(valores_previstos))

fig = px.scatter(
    comparacao_tipo_area_amanha_plot,
    x="real",
    y="previsto",
    color="tipo_area",
    opacity=0.5,
    title="Temperatura real vs prevista por tipo de área — amanhã",
    labels={
        "real": "Temperatura real (°C)",
        "previsto": "Temperatura prevista (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        name="Linha ideal"
    )
)

fig.show()

## 18. Erro por mês e tipo de área

A análise mensal mostra se o modelo erra mais em determinados períodos do ano.

Isso é útil para detectar sazonalidade ou meses de maior instabilidade climática.

In [ ]:
pred_amanha_ctx = spark.sql("""
    SELECT
        *,
        MONTH(data_formatada) AS mes
    FROM pred_amanha_ctx
""")

pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

erro_mes_tipo_area_amanha = spark.sql("""
    SELECT
        mes,
        tipo_area,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_amanha_ctx
    GROUP BY mes, tipo_area
    ORDER BY mes, tipo_area
""")

erro_mes_tipo_area_amanha.createOrReplaceTempView("erro_mes_tipo_area_amanha")
erro_mes_tipo_area_amanha.show(100, truncate=False)

In [ ]:
erro_mes_tipo_area_amanha_plot = spark_df_para_dicts(erro_mes_tipo_area_amanha)

fig = px.line(
    erro_mes_tipo_area_amanha_plot,
    x="mes",
    y="mae",
    color="tipo_area",
    markers=True,
    title="MAE por mês e tipo de área — previsão de amanhã",
    labels={
        "mes": "Mês",
        "mae": "MAE (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.show()

### Insight esperado

Se o erro aumentar em meses de transição, como outono e primavera, isso pode indicar que o modelo tem mais dificuldade em períodos de maior variabilidade atmosférica.

Se algum tipo de área apresentar erro sistematicamente maior ao longo do ano, isso sugere que o comportamento climático daquela região é mais difícil de representar com as features atuais.

## 19. Temperatura média real vs prevista por tipo de área

Este gráfico mostra se o modelo consegue reproduzir as diferenças médias entre os tipos de área.

Ele é útil para observar se o modelo mantém coerência climática entre regiões urbanas, litorâneas, serranas e interiores.

In [ ]:
media_tipo_area_amanha = spark.sql(f"""
    SELECT
        tipo_area,
        ROUND(AVG({coluna_alvo_amanha}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media
    FROM pred_amanha_ctx
    GROUP BY tipo_area
    ORDER BY tipo_area
""")

media_tipo_area_amanha.createOrReplaceTempView("media_tipo_area_amanha")
media_tipo_area_amanha.show(truncate=False)

In [ ]:
media_tipo_area_long = spark.sql("""
    SELECT
        tipo_area,
        'Temperatura real média' AS tipo,
        temperatura_real_media AS temperatura_media
    FROM media_tipo_area_amanha

    UNION ALL

    SELECT
        tipo_area,
        'Temperatura prevista média' AS tipo,
        temperatura_prevista_media AS temperatura_media
    FROM media_tipo_area_amanha
""")

media_tipo_area_long_plot = spark_df_para_dicts(media_tipo_area_long)

fig = px.bar(
    media_tipo_area_long_plot,
    x="tipo_area",
    y="temperatura_media",
    color="tipo",
    barmode="group",
    text="temperatura_media",
    title="Temperatura média real vs prevista por tipo de área — amanhã",
    labels={
        "tipo_area": "Tipo de área",
        "temperatura_media": "Temperatura média (°C)",
        "tipo": "Série"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Espera-se que regiões de `serra_altitude` apresentem temperaturas médias menores, por causa da influência da altitude.

Áreas `urbano_metropolitano` podem apresentar médias mais altas, possivelmente relacionadas à concentração urbana e ao efeito de ilha de calor.

Regiões de `litoral` tendem a ter comportamento mais estável em alguns contextos, devido à influência oceânica e à umidade.

## 20. Erro por macro região

Além do tipo de área, também é útil observar o desempenho por macro região aproximada do estado de São Paulo.

In [ ]:
erro_macro_amanha = spark.sql(f"""
    SELECT
        macro_regiao_sp,
        COUNT(*) AS total_registros,
        ROUND(AVG({coluna_alvo_amanha}), 2) AS temperatura_real_media,
        ROUND(AVG(prediction), 2) AS temperatura_prevista_media,
        ROUND(AVG(erro_absoluto), 4) AS mae
    FROM pred_amanha_ctx
    GROUP BY macro_regiao_sp
    ORDER BY macro_regiao_sp
""")

erro_macro_amanha.createOrReplaceTempView("erro_macro_amanha")
erro_macro_amanha.show(truncate=False)

In [ ]:
erro_macro_amanha_plot = spark_df_para_dicts(erro_macro_amanha)

fig = px.bar(
    erro_macro_amanha_plot,
    x="macro_regiao_sp",
    y="mae",
    text="mae",
    title="Erro médio absoluto por macro região — previsão de amanhã",
    labels={
        "macro_regiao_sp": "Macro região",
        "mae": "MAE (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 21. Relação entre altitude e erro

A altitude é uma variável importante para temperatura.

Este gráfico ajuda a observar se o modelo tem maior dificuldade em regiões mais altas, como áreas de serra.

Para manter o gráfico leve, a consulta usa uma amostra limitada.

In [ ]:
altitude_erro = spark.sql("""
    SELECT
        altitude,
        tipo_area,
        erro_absoluto
    FROM pred_amanha_ctx
    WHERE altitude IS NOT NULL
      AND tipo_area IS NOT NULL
      AND erro_absoluto IS NOT NULL
    LIMIT 5000
""")

altitude_erro_plot = spark_df_para_dicts(altitude_erro)

fig = px.scatter(
    altitude_erro_plot,
    x="altitude",
    y="erro_absoluto",
    color="tipo_area",
    opacity=0.5,
    title="Relação entre altitude e erro absoluto — previsão de amanhã",
    labels={
        "altitude": "Altitude (m)",
        "erro_absoluto": "Erro absoluto (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.show()

### Insight esperado

Caso os erros aumentem em altitudes maiores, isso pode indicar que regiões serranas possuem comportamento térmico mais específico.

Mesmo com a variável `altitude` presente no modelo, fatores locais como relevo, cobertura vegetal e massas de ar podem influenciar a temperatura de maneira mais complexa.

## 22. Salvamento das predições, métricas e coeficientes

As saídas do notebook são salvas para comparação posterior com outros modelos, como Random Forest e Redes Neurais.

In [ ]:
predicoes_lr_amanha_saida = spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        {coluna_alvo_amanha},
        prediction,
        erro_absoluto
    FROM pred_amanha_ctx
""")

predicoes_lr_semana_saida = spark.sql(f"""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        {coluna_alvo_semana},
        prediction,
        erro_absoluto
    FROM pred_semana_ctx
""")

predicoes_lr_amanha_saida.write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_lr_amanha"
)

predicoes_lr_semana_saida.write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_lr_semana"
)

metricas_lr.write.mode("overwrite").parquet(
    f"{resultados_path}/metricas_linear_regression"
)

coef_amanha.write.mode("overwrite").parquet(
    f"{resultados_path}/coeficientes_lr_amanha"
)

coef_semana.write.mode("overwrite").parquet(
    f"{resultados_path}/coeficientes_lr_semana"
)

print("Predições, métricas e coeficientes salvos com sucesso.")

## 23. Salvamento dos modelos treinados

Os modelos são salvos para que possam ser reutilizados sem necessidade de novo treinamento.

In [ ]:
modelo_lr_amanha.write().overwrite().save(
    f"{modelos_path}/linear_regression_amanha"
)

modelo_lr_semana.write().overwrite().save(
    f"{modelos_path}/linear_regression_semana"
)

print("Modelos Linear Regression salvos com sucesso.")

## 24. Conclusão

Neste notebook foram treinados dois modelos **Linear Regression** com **PySpark MLlib**:

- um modelo para prever a temperatura média de amanhã;
- um modelo para prever a temperatura média dos próximos 7 dias.

As bases utilizadas vieram diretamente do pré-processamento, já com:

- split temporal;
- imputação de valores ausentes;
- indexação de variáveis categóricas;
- features temporais de defasagem e janelas móveis.

A avaliação foi feita com **MAE**, **RMSE** e **R²**.

Além disso, foram gerados gráficos com Plotly para analisar:

- comparação geral das métricas;
- coeficientes do modelo;
- erro por tipo de área;
- real vs previsto por tipo de área;
- erro por mês;
- temperatura média real vs prevista;
- erro por macro região;
- relação entre altitude e erro.

A análise por `tipo_area` permite observar diferenças entre áreas urbanas/metropolitanas, litorâneas, serranas e interiores.

A Regressão Linear não entende sequência temporal automaticamente. Por isso, as features de defasagem e janelas móveis criadas no pré-processamento foram fundamentais para fornecer contexto temporal ao modelo.

Este modelo também funciona como uma base interpretável para comparar com modelos mais complexos, como Random Forest e Redes Neurais.